# How fast a run went, and what limited it

This notebook reads what `detect.py` writes while it scores (`_log.txt`, `_status.json` in every
task folder) and, if you saved one, Slurm's accounting listing, and shows

1. one line per task: tiles, hours, tiles per minute, download time, what was busy,
2. speed and download time over time, task by task (a slow node stands out at once),
3. which limit each task ran into: the node's scoring or the downloads,
4. the jobs: node, hours, core-hours, tiles scored per core-hour,
5. the run's totals, and what the whole plan would cost at this pace (README B4's table, live).

It only reads small text files, so it is fine on Goethe's login node, and it works on a laptop
copy of the run. The numbers come from [`performance.py`](performance.py), which prints the same
tables without Jupyter.

**On Goethe**, after the login lines of README B3 (`$RUN` set, the venv on):

```bash
python panel_pipeline/checks/performance.py          # the tables; finds the run in $RUN and asks sacct for the jobs
```

The notebook itself runs there through VS Code's Jupyter (Remote-SSH), once the venv has a kernel:
`uv pip install ipykernel` in the venv, one time. It reads `$RUN` too, so the first cell needs no editing.

**On a laptop**, with a copy of the run under `tests_output/`: the progress logs, status files and
the jobs listing are small, and this fetches just them (the jobs listing exists once the notebook
or `performance.py` has run on Goethe, or after the `sacct` line in the module docstring):

```bash
rsync -av --include='*/' --include='_log.txt' --include='_status.json' --include='jobs.txt' --exclude='*'     <user>@goethe.hhlr-gu.de:/work/<group>/<user>/runs/<run>/ tests_output/<run>/
```

In [ ]:
# ---- settings: the only cell you normally edit --------------------------------------
import os
from pathlib import Path

RUN = Path(os.environ.get("RUN", "tests_output/aca_full_2026-10-23"))   # $RUN on Goethe; else the laptop copy
JOBS = None                                       # None: <run>/jobs.txt, or sacct itself on the cluster; or a path
CLUSTER_TZ = "Europe/Berlin"                      # sacct prints the cluster's local time
CORES_PER_NODE = 40                               # Goethe general1
NODES = 3                                         # how many nodes a run would use at once (B4: ~3 fit Earth Engine's 40 requests)
EECU_PER_TILE = {"quarterly": 145, "monthly": 50} # Earth Engine cost per tile-period, EECU-seconds (B4: 57-145 measured for a quarter)
QUOTA_EECU_HOURS = 100_000                        # the project's monthly Earth Engine quota (Partner tier)
ALLOCATION_CORE_HOURS = 200_000                   # the Goethe allocation

In [ ]:
from __future__ import annotations

import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "panel_pipeline").is_dir() else Path.cwd().parents[1]   # run from the repo root or from checks/
sys.path.insert(0, str(ROOT / "panel_pipeline" / "checks"))
sys.path.insert(0, str(ROOT / "panel_pipeline" / "lib"))
import performance as perf

RUN = (ROOT / RUN) if not RUN.is_absolute() else RUN
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 1)

lines = perf.progress(RUN)                                   # every progress line of every task
sessions = perf.sessions(RUN, lines)                         # one row per detect.py session
jobs, note = perf.find_jobs(RUN, sessions, JOBS, CLUSTER_TZ)   # a saved listing, or sacct on the cluster
sessions = perf.assign_jobs(sessions, jobs)                  # which job (node) ran each session
tasks = perf.task_summary(sessions)
print(f"{RUN}\n{len(lines):,} progress lines, {len(sessions)} sessions, {len(tasks)} tasks\n{note}")

## 1. One line per task

`tiles_per_min` is tiles scored divided by the hours the task's sessions ran (what you get per task,
tails included); `rate` is the median of the progress lines (what it does while it is going).
`dl_median_s` is the median download time, `workers_busy` and `scoring_busy` the medians of the
progress lines, and `limited_by` reads them: **scoring busy near 100 % means the node is the
limit**; workers busy near 100 % with scoring well below means Earth Engine, or this node's link
to it, is. `nodes` needs the jobs listing.

In [ ]:
columns = ["frequency", "period", "subregion", "tiles_saved", "tiles_planned", "failed", "complete",
           "sessions", "hours", "tiles_per_min", "rate", "dl_median_s", "workers_busy", "scoring_busy",
           "retries", "workers", "limit", "nodes", "limited_by"]
tasks[columns] if len(tasks) else "no task folder found under " + str(RUN / "detect")

## 2. Speed and download time over time

One line per task. A task that starts slow is loading its model; one that stays slow with long
downloads sits on a slow node (28 Oct 2026: node22-006 downloaded at 18-35 s while the other
nodes took 5 s). The drop at the end of a node's work is the tail: the last task or two of a task
file run alone with their 10 threads while the rest of the node idles.

In [ ]:
busy = lines[lines["rate"] > 0]
if len(busy):
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    for task, g in busy.groupby("task", sort=False):
        axes[0].plot(g["time"], g["rate"], ".-", ms=3, lw=0.8, label=task)
        axes[1].plot(g["time"], g["dl_median_s"], ".-", ms=3, lw=0.8)
    axes[0].set_ylabel("tiles / min (one task)"); axes[0].grid(alpha=0.3)
    axes[1].set_ylabel("download median, s"); axes[1].grid(alpha=0.3)
    axes[0].legend(ncol=4, fontsize=8, loc="upper left")
    axes[1].set_xlabel("UTC"); fig.autofmt_xdate(); plt.tight_layout()
else:
    print("no progress lines: the _log.txt files are missing from this copy of the run")

## 3. What was the limit

Every progress line, workers busy against scoring busy. Points in the top-left corner: the node's
scoring is the limit (more download workers would not help; more tasks of fewer threads might).
Bottom-right: the downloads are (more workers per task, or a different node). Top-right: both,
which is where a well-sized job sits.

In [ ]:
if len(busy):
    fig, ax = plt.subplots(figsize=(6, 6))
    for task, g in busy.groupby("task", sort=False):
        ax.scatter(g["workers_busy"], g["scoring_busy"], s=8, alpha=0.5, label=task)
    ax.set_xlabel("workers busy, % (downloads)"); ax.set_ylabel("scoring busy, % (the node)")
    ax.set_xlim(0, 105); ax.set_ylim(0, 105); ax.grid(alpha=0.3)
    ax.axhline(80, ls=":", c="grey"); ax.axvline(80, ls=":", c="grey")
    ax.legend(fontsize=7, ncol=2, loc="lower left"); plt.tight_layout()
    print(tasks["limited_by"].value_counts().to_string())

## 4. Jobs, nodes and core-hours

Needs the jobs listing (found by itself on Goethe; see the top). Each session is matched to the job that was running when it started, so
a job's `tiles` are what its tasks scored and `tiles_per_core_hour` is the number to plan with.
A job with far fewer tiles per core-hour than the others at the same settings is a slow node.

In [ ]:
job_table = perf.job_summary(sessions, jobs, CORES_PER_NODE)
if len(job_table):
    display(job_table)
    by_node = (job_table[job_table["tiles"] > 0].groupby("node")[["hours", "core_hours", "tiles"]].sum())
    by_node["tiles_per_core_hour"] = by_node["tiles"] / by_node["core_hours"]
    display(by_node.sort_values("tiles_per_core_hour"))
else:
    print("no jobs listing: run this on Goethe once, or copy jobs.txt (top of the notebook), for core-hours and node names")

## 5. Totals, and what the full plan would cost at this pace

The projection takes the run's own plan (`plan/tiles.csv`, the periods of `run/config.yaml`) and
the measured tile-periods per core-hour (from the jobs listing; without one, an estimate from the
tasks' own hours, which leaves out the idle tail of each node). `EECU_PER_TILE` at the top is the
Earth Engine cost; the real figure for a month is on the Earth Engine configuration page
(console.cloud.google.com/earth-engine/configuration), divided by the tile-periods scored.

In [ ]:
import csv
from config import load_config

t = perf.totals(tasks, job_table, CORES_PER_NODE)
for key, value in t.items():
    print(f"{key:32} {value:,.1f}" if isinstance(value, float) else f"{key:32} {value:,}")

per_core_hour = t.get("tile_periods_per_core_hour")
if not per_core_hour and t.get("task_hours"):
    per_core_hour = t["tile_periods_saved"] / (t["task_hours"] * CORES_PER_NODE / 4)   # 4 tasks share a node
    print(f"\nno jobs listing: tile-periods per core-hour estimated from the tasks' hours as {per_core_hour:,.0f} (optimistic)")

if per_core_hour and (RUN / "run" / "config.yaml").is_file():
    cfg = load_config(RUN / "run" / "config.yaml")
    with open(RUN / "plan" / "tiles.csv", newline="") as f:
        n_tiles = sum(1 for _ in csv.DictReader(f))
    n_periods = {freq: len(periods) for freq, periods in cfg.periods.items()}
    print(f"\nthis run's plan: {n_tiles:,} tiles x {n_periods} periods")
    proj = perf.projection(n_tiles, n_periods, per_core_hour, EECU_PER_TILE, CORES_PER_NODE, NODES,
                           QUOTA_EECU_HOURS, ALLOCATION_CORE_HOURS)
    display(proj.style.format({"tile_periods": "{:,.0f}", "core_hours": "{:,.0f}", "share_of_allocation": "{:.0%}",
                               "node_days": "{:,.0f}", f"days_on_{NODES}_nodes": "{:,.0f}",
                               "eecu_hours": "{:,.0f}", "months_of_quota": "{:.1f}"}))

**Reading the table.** `core_hours` is what the scope costs of the Goethe allocation at this run's
pace; `days_on_N_nodes` is the calendar with N nodes at once (about three fit Earth Engine's 40
concurrent requests); `months_of_quota` is how many months of the Earth Engine quota the scope
needs at `EECU_PER_TILE`. Over the quota the project runs in restricted mode (slower), so a scope
above one month of quota either runs slower, is spread over months, or shrinks.